# **Discharge DataSet**
### Ève Castonguay, LIRA

In [2]:
# This code creates an Xarray DataSet out of all relevant swot and grdc data (discharge measures, lon, lat, etc.) and then shows various plots for statistical analysis.
# The main goal is to compare the swot discharge data with in-situ values available in the grdc dataset.
# Author: Ève Castonguay, LIRA (CNRS)
# Creation date: 2026-05-29 [YYYY-MM-DD]
# Version 0.1: AAAA-MM-JJ

In [1]:
# Imports
from datetime import datetime
import os
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import glob 
import numpy as np
import netCDF4 as nc
from scipy.spatial import KDTree
from mpl_toolkits.basemap import Basemap
import cartopy.crs as ccrs
import math
from scipy import stats
import matplotlib.colors as mcolors
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import geopy.distance
import random
import re
from rapidfuzz import fuzz

# Utils file for external functions, with auto-reload
%load_ext autoreload
%autoreload 2
import utils

## **Part I. Create the DataSet**

### **V3** Correspondence based on 1) distance, 2) area & name

##### This version uses the watershed long_pp & lat_pp

In [ ]:
version = 'v3_1'
## Section 1 : Setting some variables
# 1.1 Continents
continent_list = ['na', 'af', 'as', 'eu', 'sa', 'oc']
# 1.2 Period during which SWOT has data (2023-03-29 to 2025-05-02)
swot_start = '2023-03-29'
swot_end = '2025-05-02'
# 1.3 Empty dictionnaries for the DataArrays (there probably is a better way?)
# grdc
runoff_darrays_g = {} 
geox_darrays_g = {}
geoy_darrays_g = {}
area_darrays_g = {}
river_darrays_g = {}
country_darrays_g = {}
# swot
dschg_darrays_s = {}
geox_darrays_s = {}
geoy_darrays_s = {}
id_darrays_s = {}
w_darrays_s = {} # from sword
area_darrays_s = {} # from sword
r_id_up_darrays_s = {} # from sword
r_id_dn_darrays_s = {} # from sword
river_darrays_s = {} # from sword

## Section 2 : Loop over continents 
for i_continent in continent_list: 
    
    ## Section 3 : Swot extraction
    # 3.1 Reading
    dir_l4 = "/obs/ecastonguay/swot_data/L4_discharge/"
    file_suffix = "_sword_v16_SOS_results_unconstrained_20230502T204408_20250502T204408_20251219T163700.nc"
    single_file_name = dir_l4 + i_continent + file_suffix 
    data_swot = nc.Dataset(single_file_name) # level 4 satellite data
    # 3.2 Extracting
    r_id_swt = data_swot.groups['reaches']['reach_id'][:]
    time_swt = data_swot.groups["consensus"]['time_int'][:]
    geox_swt = data_swot.groups["reaches"]['x'][:]
    geoy_swt = data_swot.groups["reaches"]['y'][:]
    dschg_swt = data_swot.groups["consensus"]['consensus_q'] # shape (38048,). if no data at a reach, contains array([-1.e+12]). else. contains array() with 766 data
    assert dschg_swt.shape == r_id_swt.shape # dschhg is a list of arrays, same len as r_ids
    # 3.3 In DataArrays (for better access)
    geox_darray_full_s = xr.DataArray( 
        data=geox_swt,
        dims=["reach_id"], 
        coords=dict(reach_id=r_id_swt,),
        name="x coordinate swot")
    geoy_darray_full_s = xr.DataArray(
        data=geoy_swt,
        dims=["reach_id"], 
        coords=dict(reach_id=r_id_swt,),
        name="y coordinate swot")
    """dschg_darray_full_s = xr.DataArray( 
        data=dschg_swt,
        dims=["reach_id","time"], # name of the dimensions
        coords=dict(
            reach_id=r_id_swt,
            time=time_swt,
        ),
        attrs=dict(
            description="Consensus_q from SWOT",
            units="m3/s",
            missing_value=mv
        ),
        name="swot discharge"
    )"""

    ## Section 4 : Sword extraction
    # 4.1 Reading
    dir_swr = "/obs/ecastonguay/sword_data/netcdf_v16" # sword v16
    file_swr = i_continent + "_sword_v16.nc"
    path_swr = os.path.join(dir_swr,file_swr)
    data_swr = nc.Dataset(path_swr) # open the netcdf file
    # 4.2 Extracting
    r_id_swr = data_swr["reaches"]["reach_id"][:]
    facc_swr = data_swr["reaches"]["facc"][:]
    river_swr = data_swr["reaches"]["river_name"][:]
    width_swr = data_swr["reaches"]["width"][:]
    # 4.3 In DataArrays
    facc_darray_full_s = xr.DataArray(
        data=facc_swr,
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="facc sword") 
    river_darray_full_s = xr.DataArray(
        data=river_swr, # contains ndarrays()
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 
    width_darray_full_s = xr.DataArray(
        data=width_swr,
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 

    ## Section 5 : Grdc extraction
    # 5.1 Reading
    dir_grdc_prefix = "/obs/ecastonguay/grdc_data/"
    file_nc = i_continent + ".nc"
    path_nc = os.path.join(dir_grdc_prefix,i_continent,file_nc)
    file_json = "stationbasins_" + i_continent + ".geojson"
    path_json = os.path.join(dir_grdc_prefix,i_continent,file_json)
    # 5.2 Extracting : discharge file
    data_grdc = xr.open_dataset(path_nc, engine="netcdf4") # <xarray.Dataset>
    data_23_25_g = data_grdc.sel(time=slice(swot_start,swot_end)) # DataSet. slice here includes the last day
    data_23_25_g = data_23_25_g.transpose() # swap dimensions to have (id,time) instead of (time,id)
    # 5.3 In DataArrays : discharge file
    dschg_darray_g = data_23_25_g['runoff_mean']      
    country_darray_g = data_23_25_g['country'] 
    # list of the stations_id of grdc  
    list_station_id = dschg_darray_g["id"].values 
    # 5.4 Extracting : watershed file
    data_ws = gpd.read_file(path_json) # watershed
    # pandas series
    station_id_gdf = data_ws['grdc_no'] 
    area_gdf = data_ws['area_calc']
    river_gdf = data_ws['river']
    geox_gdf = data_ws['long_pp']
    geoy_gdf = data_ws['lat_pp']
    # 5.5 In DataArrays : watershed file
    area_darray_g = xr.DataArray(
        data=area_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="Watershed areas (km2)"),
        name="watershed areas")
    river_darray_g = xr.DataArray(
        data=river_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="GRDC river names"),
        name="river names")
    geox_darray_g = xr.DataArray(
        data=geox_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="x coordinates of the GRDC station"),
        name="geo x grdc")
    geoy_darray_g = xr.DataArray(
        data=geoy_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="y coordinates of the GRDC station"),
        name="geo y grdc")  

    ## Section 6 : Empty ndarrays for swot/sword data 
    time_dim = pd.date_range(start=swot_start, end=swot_end, freq='D') # 766. ndarray-like of datetime64 data. dtype='datetime64[us].
    time_dim_len = len(time_dim)
    # 6.1 Swot
    # discharge
    dschg_ndarray = np.full((len(list_station_id), time_dim_len), np.nan, dtype=np.float64) # dim (id, time)
    # lon
    x_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # lat
    y_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id
    r_id_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # 6.2 Sword
    # width
    w_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # area
    facc_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id up
    r_id_up_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id down
    r_id_dn_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # river name
    river_ndarray = np.full((len(list_station_id)), '', dtype=object) # dim (id)

    ## Section 7 : Match by distance (k nearest neighbors)
    station_pos = 0
    no_match_area = 0 # n of stations that werent matched due to area diff. (corresp function)
    no_match_name = 0 # n of stations that werent matched due to name diff. (corresp function)
    no_match_corresp = 0 # n of stations that werent matched due to corresp function
    no_match_ws = 0 # n of stations that werent matched due to (y,x) watershed data
    for station_id in list_station_id:
        
        # 7.1 Target (lon,lat) from grdc
        if station_id not in geox_darray_g.id.values:
            station_pos += 1
            no_match_ws += 1
            continue
        x_found_station = geox_darray_g.sel(id=station_id).values # 4977015 -> "not all values found in index 'id'. Try setting the `method` keyword argument (example: method='nearest')."
        y_found_station = geoy_darray_g.sel(id=station_id).values # [tested]

        # 7.2 List of reach coordinates in swot
        x_coordinates_values = geox_darray_full_s.values
        y_coordinates_values = geoy_darray_full_s.values
        
        # 7.3 K-D tree to search nearest neighbor (looking for the reach with the closest coordinates to the station)
        k_neighbors = 10
        stacked_xy = np.vstack((x_coordinates_values,y_coordinates_values)).T
        distance_list, index_list = KDTree(stacked_xy).query([x_found_station, y_found_station],k=k_neighbors) # we take the k nearest neighboors so that if the nearest reach doesn't contain swot data, we look at the second nearest, ...
        no_r_found = False
        for ii in range(k_neighbors): 

            # 7.4 Info of the nearest found reach
            distance = distance_list[ii] # distance
            r_index = index_list[ii] # index 
            r_id = r_id_swt[r_index] # r_id

            # 7.5 Correspondence
            result, reason = utils.corresp_area_name(station_id, r_id, r_index, area_darray_g, river_darray_g, facc_darray_full_s, river_darray_full_s, dschg_swt)
            if result is not None:
                sel_r_id, sel_r_index, dschg_flt_s, mask_mv, name_river_s = result 
                break
            if reason == 'area':
                no_match_area += 1
            if reason == 'name':
                no_match_name += 1
            
            # 7.6 No correspondence after last neighbor
            if ii == k_neighbors-1:
                no_r_found = True

        if no_r_found:
            station_pos += 1 
            no_match_corresp += 1
            continue # moving on to the next station, there was no reach found for this station_id. data will stay NaN.
        else: 
            ## Section 8 : Filling empty ndarrays with sword/swot data
            # 8.1 Using the time data to find the indexes to fill the empty ndarrays with data 
            time_s = time_swt[sel_r_index]
            time_flt_s = time_s[mask_mv] # RENDUE ICI
            assert dschg_flt_s.shape == time_flt_s.shape # raises assertion error if not true
            # converting time from int to datetime
            epoch = np.datetime64('2000-01-01')
            datetime_flt_s = epoch + time_flt_s.astype('timedelta64[s]')
            # put hours,min,sec to 00:00:00 to ignore time and only keep date
            datetime_norm = pd.to_datetime(datetime_flt_s).normalize()
            # get index 
            ndarray_swot_index = time_dim.get_indexer(datetime_norm) # indice of everywhere where the day corresponds
            if -1 in ndarray_swot_index:
                print("Error: [6.1] Not all dates from the SWOT data were found in the time_dim vector (.get_indexer method)") 

            # 8.2 Coordinates
            sel_r_x = geox_darray_full_s.sel(reach_id=sel_r_id)
            sel_r_y = geoy_darray_full_s.sel(reach_id=sel_r_id)
            x_ndarray[station_pos] = sel_r_x
            y_ndarray[station_pos] = sel_r_y
            
            # 8.3 Discharge
            dschg_ndarray[station_pos,ndarray_swot_index] = dschg_flt_s # [tested] where there is data the space will be filled, otherwise stays NaN
            
            # 8.4 Reach id
            r_id_ndarray[station_pos] = sel_r_id

            # 8.5 Width, facc, river name (here I assume that a reach id in swot will be found in the sword database also; this assumption never caused me problems)
            w_ndarray[station_pos] = width_darray_full_s.sel(reach_id=sel_r_id)
            facc_ndarray[station_pos] = facc_darray_full_s.sel(reach_id=sel_r_id)
            river_ndarray[station_pos] = name_river_s
        
        station_pos += 1 # end of the (for station_id in list_station_id:) loop

    assert station_pos == len(list_station_id) # at the end of the station loop, these should be equal

    print(f"Number of reaches that weren't matched due to corresp. algo: {no_match_corresp} for the {i_continent} continent.")
    print(f"Number of reaches that weren't matched due to watershed data: {no_match_ws} for the {i_continent} continent.")
    print(f"Number of no match due to area differences: {no_match_area} for the {i_continent} continent (*multiple trys for a single reach*).")
    print(f"Number of no match due to name differences: {no_match_name} for the {i_continent} continent (*multiple trys for a single reach*).")

    
    ## Section 9 : Create DataArrays for SWOT/SWORD (ndarrays ordered by station_id)
    # 9.1 Discharge
    dschg_darray_s = xr.DataArray(
        data=dschg_ndarray,
        dims=["id","time"], # name of the dimensions
        coords=dict(
            id=list_station_id,
            time=time_dim,
        ),
        attrs=dict(
            description="Consensus_q from SWOT",
            units="m3/s",
        ),
        name="swot discharge"
    )
    # 9.2 Coordinates
    if len(x_ndarray) != len(y_ndarray):
        print(f"Error: [7.2] The x and y vector of all the selected reaches' coordinates doesn't match in size")
        #break # *** activate when continent loop is added!
    # x_y_ndarray = np.hstack((x_ndarray,y_ndarray)) # doesn't work for now
    # dataarrays
    geox_darray_s = xr.DataArray(
        data=x_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="X coordinate of each selected reach in the SWOT data",
            units="degrees",
        ),
        name="swot x coordinate"
    )
    geoy_darray_s = xr.DataArray(
        data=y_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Y coordinate of each selected reach in the SWOT data",
            units="degrees",
        ),
        name="swot y coordinate"
    )
    # 9.3 Reach ids
    id_darray_s = xr.DataArray(
        data=r_id_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Id's of the selected reaches in the SWOT data"
        ),
        name="swot reach ids"
    )
    # 9.4 Width
    w_darray_s = xr.DataArray(
        data=w_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Average width for a SWOT reach (units: meters)."
        ),
        name="sword widths"
    )
    # 9.5 Area
    area_darray_s = xr.DataArray(
        data=facc_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Maximum flow accumulation value for a node or reach. Flow accumulation values are extracted from the MERIT Hydro dataset (Yamazaki et al., 2019) (units: square kilometers)."
        ),
        name="sword area" 
    )
    # 9.6 River name
    river_darray_s = xr.DataArray(
        data=river_ndarray.astype(str),
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="All river names associated with a node or reach. If there are multiple names for a node or reach they are listed in alphabetical order and separated by a semicolon."
        ),
        name="sword river name" 
    )

    ## Section 10 : Make sure all grdc DataArrays have time dim of 766 days (between 2023-03-29 & 2025-05-02)
    if len(data_23_25_g.time) != len(dschg_darray_g.time):
        print(f"Error: [8.1] Both arrays are assumed to match in size. Please verify the code.")
    if len(dschg_darray_g.time) != time_dim_len:
        dschg_darray_g = dschg_darray_g.reindex(time=time_dim) # asia doesn't -> fill the time between 2024 & 2025 with NaN
        print(f"[8.1] DataArray of continent {i_continent} was reindexed") # missing values seem to be nan [tested]

    ## Section 11 : Append continent DataArrays to list
    # grdc
    runoff_darrays_g[i_continent] = dschg_darray_g
    geox_darrays_g[i_continent] = geox_darray_g
    geoy_darrays_g[i_continent] = geoy_darray_g
    area_darrays_g[i_continent] = area_darray_g
    river_darrays_g[i_continent] = river_darray_g
    country_darrays_g[i_continent] = country_darray_g
    # swot
    dschg_darrays_s[i_continent] = dschg_darray_s
    geox_darrays_s[i_continent] = geox_darray_s
    geoy_darrays_s[i_continent] = geoy_darray_s
    id_darrays_s[i_continent] = id_darray_s
    w_darrays_s[i_continent] = w_darray_s
    area_darrays_s[i_continent] = area_darray_s
    river_darrays_s[i_continent] = river_darray_s

# end of the (for i_continent in continent_list) loop 

## Section 12 : Concathenate continent DataArrays. These are individual DataArrays that contain values for all continents
# grdc
runoff_global_g = xr.concat(list(runoff_darrays_g.values()), dim='id') 
geox_global_g = xr.concat(list(geox_darrays_g.values()), dim='id')
geoy_global_g = xr.concat(list(geoy_darrays_g.values()), dim='id')
area_global_g = xr.concat(list(area_darrays_g.values()), dim='id')
river_global_g = xr.concat(list(river_darrays_g.values()), dim='id')
country_global_g = xr.concat(list(country_darrays_g.values()), dim='id')
# swot
dschg_global_s = xr.concat(list(dschg_darrays_s.values()), dim='id') 
geox_global_s = xr.concat(list(geox_darrays_s.values()), dim='id')
geoy_global_s = xr.concat(list(geoy_darrays_s.values()), dim='id')
id_global_s = xr.concat(list(id_darrays_s.values()), dim='id')
width_global_s = xr.concat(list(w_darrays_s.values()), dim='id')
area_global_s = xr.concat(list(area_darrays_s.values()), dim='id')
river_global_s = xr.concat(list(river_darrays_s.values()), dim='id')

## Section 13 : Create the DataSet and append all the variables (DataArrays) to it
# grdc
dset_global = runoff_global_g.to_dataset(name='dschg_global_g')
dset_global['geox_global_g'] = geox_global_g
dset_global['geoy_global_g'] = geoy_global_g
dset_global['area_global_g'] = area_global_g
dset_global['river_global_g'] = river_global_g
dset_global['country_global_g'] = country_global_g
# swot
dset_global['dschg_global_s'] = dschg_global_s
dset_global['geox_global_s'] = geox_global_s
dset_global['geoy_global_s'] = geoy_global_s
dset_global['id_global_s'] = id_global_s
dset_global['width_global_s'] = width_global_s
dset_global['area_global_s'] = area_global_s
dset_global['river_global_s'] = river_global_s

# Change the data type of the reaches id which is (int)
# dset_global["id_global_s"] = dset_global["id_global_s"].astype(int) # *** actually, this can't be done since a numpy array of type int cannot contain nans (they would be transformed in -9223372036854775808)

# Print
print(dset_global)

# Save to netcdf
dset_global.to_netcdf("/obs/ecastonguay/scripts/global_dset_" + version + ".nc")

/obs/ecastonguay/miniconda3/envs/swot-env/lib/python3.14/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


Number of reaches that weren't matched due to corresp. algo: 1192 for the na continent.
Number of reaches that weren't matched due to watershed data: 5 for the na continent.
Number of no match due to area differences: 492 for the na continent (*multiple trys for a single reach*).
Number of no match due to name differences: 3595 for the na continent (*multiple trys for a single reach*).
Number of reaches that weren't matched due to corresp. algo: 324 for the af continent.
Number of reaches that weren't matched due to watershed data: 5 for the af continent.
Number of no match due to area differences: 76 for the af continent (*multiple trys for a single reach*).
Number of no match due to name differences: 1150 for the af continent (*multiple trys for a single reach*).
Number of reaches that weren't matched due to corresp. algo: 113 for the as continent.
Number of reaches that weren't matched due to watershed data: 0 for the as continent.
Number of no match due to area differences: 4 for t

### **V4** Correspondence based on 1) name, 2) distance & area

##### Uses long_pp & lat_pp from watershed data, but dschg data not in darray

In [ ]:
version = 'v4'
## Section 1 : Setting some variables
# 1.1 Continents
continent_list = ['na', 'af', 'as', 'eu', 'sa', 'oc']
# 1.2 Period during which SWOT has data (2023-03-29 to 2025-05-02)
swot_start = '2023-03-29'
swot_end = '2025-05-02'
# 1.3 Empty dictionnaries for the DataArrays (there probably is a better way?)
# grdc
runoff_darrays_g = {} 
geox_darrays_g = {}
geoy_darrays_g = {}
area_darrays_g = {}
river_darrays_g = {}
country_darrays_g = {}
# swot
dschg_darrays_s = {}
geox_darrays_s = {}
geoy_darrays_s = {}
id_darrays_s = {}
w_darrays_s = {} # from sword
area_darrays_s = {} # from sword
r_id_up_darrays_s = {} # from sword
r_id_dn_darrays_s = {} # from sword
river_darrays_s = {} # from sword

## Section 2 : Loop over continents 
for i_continent in continent_list: 
    
    ## Section 3 : Swot extraction
    # 3.1 Reading
    dir_l4 = "/obs/ecastonguay/swot_data/L4_discharge/"
    file_suffix = "_sword_v16_SOS_results_unconstrained_20230502T204408_20250502T204408_20251219T163700.nc"
    single_file_name = dir_l4 + i_continent + file_suffix 
    data_swot = nc.Dataset(single_file_name) # level 4 satellite data
    # 3.2 Extracting
    r_id_swt = data_swot.groups['reaches']['reach_id'][:]
    time_swt = data_swot.groups["consensus"]['time_int'][:]
    geox_swt = data_swot.groups["reaches"]['x'][:]
    geoy_swt = data_swot.groups["reaches"]['y'][:]
    dschg_swt = data_swot.groups["consensus"]['consensus_q'] # shape (38048,). if no data at a reach, contains array([-1.e+12]). else. contains array() with 766 data
    assert dschg_swt.shape == r_id_swt.shape # dschhg is a list of arrays, same len as r_ids
    # 3.3 In DataArrays (for better access)
    geox_darray_full_s = xr.DataArray( 
        data=geox_swt,
        dims=["reach_id"], 
        coords=dict(reach_id=r_id_swt,),
        name="x coordinate swot")
    geoy_darray_full_s = xr.DataArray(
        data=geoy_swt,
        dims=["reach_id"], 
        coords=dict(reach_id=r_id_swt,),
        name="y coordinate swot")
    """dschg_darray_full_s = xr.DataArray( 
        data=dschg_swt,
        dims=["reach_id","time"], # name of the dimensions
        coords=dict(
            reach_id=r_id_swt,
            time=time_swt,
        ),
        attrs=dict(
            description="Consensus_q from SWOT",
            units="m3/s",
            missing_value=mv
        ),
        name="swot discharge"
    )"""

    ## Section 4 : Sword extraction
    # 4.1 Reading
    dir_swr = "/obs/ecastonguay/sword_data/netcdf_v16" # sword v16
    file_swr = i_continent + "_sword_v16.nc"
    path_swr = os.path.join(dir_swr,file_swr)
    data_swr = nc.Dataset(path_swr) # open the netcdf file
    # 4.2 Extracting
    r_id_swr = data_swr["reaches"]["reach_id"][:]
    facc_swr = data_swr["reaches"]["facc"][:]
    river_swr = data_swr["reaches"]["river_name"][:]
    width_swr = data_swr["reaches"]["width"][:]
    # 4.3 In DataArrays
    facc_darray_full_s = xr.DataArray(
        data=facc_swr,
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="facc sword") 
    width_darray_full_s = xr.DataArray(
        data=width_swr,
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 
    river_darray_full_s = xr.DataArray(
        data=river_swr, # contains ndarrays()
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 
    # 4.4 Pretreatment on sword names
    river_clean_full_s = []
    index_clean_full_s = [] # sword
    for idx_river, long_name in enumerate(river_darray_full_s.values):
        parts = long_name.split("; ")
        for p in parts:
            river_clean_full_s.append(re.sub(r'\s*\(.*?\)', '', p).lower().strip())
            index_clean_full_s.append(idx_river)
        
    
    ## Section 5 : Grdc extraction
    # 5.1 Reading
    dir_grdc_prefix = "/obs/ecastonguay/grdc_data/"
    file_nc = i_continent + ".nc"
    path_nc = os.path.join(dir_grdc_prefix,i_continent,file_nc)
    file_json = "stationbasins_" + i_continent + ".geojson"
    path_json = os.path.join(dir_grdc_prefix,i_continent,file_json)
    # 5.2 Extracting : discharge file
    data_grdc = xr.open_dataset(path_nc, engine="netcdf4") # <xarray.Dataset>
    data_23_25_g = data_grdc.sel(time=slice(swot_start,swot_end)) # DataSet. slice here includes the last day
    data_23_25_g = data_23_25_g.transpose() # swap dimensions to have (id,time) instead of (time,id)
    # 5.3 In DataArrays : discharge file
    dschg_darray_g = data_23_25_g['runoff_mean']      
    country_darray_g = data_23_25_g['country'] 
    # list of the stations_id of grdc  
    list_station_id = dschg_darray_g["id"].values 
    # 5.4 Extracting : watershed file
    data_ws = gpd.read_file(path_json) # watershed
    # pandas series
    station_id_gdf = data_ws['grdc_no'] 
    area_gdf = data_ws['area_calc']
    river_gdf = data_ws['river']
    geox_gdf = data_ws['long_pp']
    geoy_gdf = data_ws['lat_pp']
    # 5.5 In DataArrays : watershed file
    area_darray_g = xr.DataArray(
        data=area_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="Watershed areas (km2)"),
        name="watershed areas")
    river_darray_g = xr.DataArray(
        data=river_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="GRDC river names"),
        name="river names")
    geox_darray_g = xr.DataArray(
        data=geox_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="x coordinates of the GRDC station"),
        name="geo x grdc")
    geoy_darray_g = xr.DataArray(
        data=geoy_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="y coordinates of the GRDC station"),
        name="geo y grdc")  

    ## Section 6 : Empty ndarrays for swot/sword data 
    time_dim = pd.date_range(start=swot_start, end=swot_end, freq='D') # 766. ndarray-like of datetime64 data. dtype='datetime64[us].
    time_dim_len = len(time_dim)
    # 6.1 Swot
    # discharge
    dschg_ndarray = np.full((len(list_station_id), time_dim_len), np.nan, dtype=np.float64) # dim (id, time)
    # lon
    x_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # lat
    y_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id
    r_id_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # 6.2 Sword
    # width
    w_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # area
    facc_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id up
    r_id_up_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id down
    r_id_dn_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # river name
    river_ndarray = np.full((len(list_station_id)), '', dtype=object) # dim (id)

    ## Section 7 : Loop on stations
    station_pos = 0
    no_match_dist_area = 0 # n of stations that werent matched due to area diff. (corresp function)
    no_match_name = 0 # n of stations that werent matched due to name diff. (corresp function)
    no_match_dschg = 0 # n of stations that werent matched due to empty discharge data (corresp function)
    no_match_corresp = 0 # n of stations that werent matched due to corresp function
    no_match_ws = 0 # n of stations that werent matched due to (y,x) watershed data
    for station_id in list_station_id:
        
        # 7.1 Station id in watershed data?
        if station_id not in geox_darray_g.id.values:
            station_pos += 1
            no_match_ws += 1
            continue
        
        # 7.2 Coordinates of station
        x_found_station = geox_darray_g.sel(id=station_id).values # 4977015 -> "not all values found in index 'id'. Try setting the `method` keyword argument (example: method='nearest')."
        y_found_station = geoy_darray_g.sel(id=station_id).values # [tested]

        ## Section 8 : Corresp. based on names, then distance and area
        result, reason = utils.corresp_name_dist_area_v4(station_id, river_clean_full_s, index_clean_full_s, r_id_swr, r_id_swt, area_darray_g, river_darray_g, facc_darray_full_s, dschg_swt, x_found_station, y_found_station, geox_darray_full_s, geoy_darray_full_s)

        if result is not None:
            sel_r_id, sel_index_swt, dschg_flt_s, mask_mv, name_river_s = result 
        else: 
            if reason == 'no match on name':
                no_match_name += 1
            if reason == 'no match on distance or area':
                no_match_dist_area += 1
            if reason == 'empty dschg data':
                no_match_dschg += 1
            no_match_corresp += 1
            station_pos += 1 
            continue

        ## Section 8 : Filling empty ndarrays with sword/swot data
        # 8.1 Using the time data to find the indexes to fill the empty ndarrays with data 
        time_s = time_swt[sel_index_swt]
        time_flt_s = time_s[mask_mv] 
        assert dschg_flt_s.shape == time_flt_s.shape # raises assertion error if not true
        # converting time from int to datetime
        epoch = np.datetime64('2000-01-01')
        datetime_flt_s = epoch + time_flt_s.astype('timedelta64[s]')
        # put hours,min,sec to 00:00:00 to ignore time and only keep date
        datetime_norm = pd.to_datetime(datetime_flt_s).normalize()
        # get index 
        ndarray_swot_index = time_dim.get_indexer(datetime_norm) # indice of everywhere where the day corresponds
        if -1 in ndarray_swot_index:
            print("Error: [6.1] Not all dates from the SWOT data were found in the time_dim vector (.get_indexer method)") 

        # 8.2 Coordinates
        sel_r_x = geox_darray_full_s.sel(reach_id=sel_r_id)
        sel_r_y = geoy_darray_full_s.sel(reach_id=sel_r_id)
        x_ndarray[station_pos] = sel_r_x
        y_ndarray[station_pos] = sel_r_y
        
        # 8.3 Discharge
        dschg_ndarray[station_pos,ndarray_swot_index] = dschg_flt_s # [tested] where there is data the space will be filled, otherwise stays NaN
        
        # 8.4 Reach id
        r_id_ndarray[station_pos] = sel_r_id

        # 8.5 Width, facc, river name (here I assume that a reach id in swot will be found in the sword database also; this assumption never caused me problems)
        w_ndarray[station_pos] = width_darray_full_s.sel(reach_id=sel_r_id)
        facc_ndarray[station_pos] = facc_darray_full_s.sel(reach_id=sel_r_id)
        river_ndarray[station_pos] = name_river_s
        
        station_pos += 1 # end of the (for station_id in list_station_id:) loop

    assert station_pos == len(list_station_id) # at the end of the station loop, these should be equal

    print(f"Number of reaches that weren't matched due to corresp. algo: {no_match_corresp} for the {i_continent} continent \n{no_match_name} for: no match on name \n{no_match_dist_area} for: no match on area/distance \n{no_match_dschg} for: empty discharge")
    print(f"Number of reaches that weren't matched due to watershed data: {no_match_ws} for the {i_continent} continent.")

    ## Section 9 : Create DataArrays for SWOT/SWORD (ndarrays ordered by station_id)
    # 9.1 Discharge
    dschg_darray_s = xr.DataArray(
        data=dschg_ndarray,
        dims=["id","time"], # name of the dimensions
        coords=dict(
            id=list_station_id,
            time=time_dim,
        ),
        attrs=dict(
            description="Consensus_q from SWOT",
            units="m3/s",
        ),
        name="swot discharge"
    )
    # 9.2 Coordinates
    if len(x_ndarray) != len(y_ndarray):
        print(f"Error: [7.2] The x and y vector of all the selected reaches' coordinates doesn't match in size")
        #break # *** activate when continent loop is added!
    # x_y_ndarray = np.hstack((x_ndarray,y_ndarray)) # doesn't work for now
    # dataarrays
    geox_darray_s = xr.DataArray(
        data=x_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="X coordinate of each selected reach in the SWOT data",
            units="degrees",
        ),
        name="swot x coordinate"
    )
    geoy_darray_s = xr.DataArray(
        data=y_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Y coordinate of each selected reach in the SWOT data",
            units="degrees",
        ),
        name="swot y coordinate"
    )
    # 9.3 Reach ids
    id_darray_s = xr.DataArray(
        data=r_id_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Id's of the selected reaches in the SWOT data"
        ),
        name="swot reach ids"
    )
    # 9.4 Width
    w_darray_s = xr.DataArray(
        data=w_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Average width for a SWOT reach (units: meters)."
        ),
        name="sword widths"
    )
    # 9.5 Area
    area_darray_s = xr.DataArray(
        data=facc_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Maximum flow accumulation value for a node or reach. Flow accumulation values are extracted from the MERIT Hydro dataset (Yamazaki et al., 2019) (units: square kilometers)."
        ),
        name="sword area" 
    )
    # 9.6 River name
    river_darray_s = xr.DataArray(
        data=river_ndarray.astype(str),
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="All river names associated with a node or reach. If there are multiple names for a node or reach they are listed in alphabetical order and separated by a semicolon."
        ),
        name="sword river name" 
    )

    ## Section 10 : Make sure all grdc DataArrays have time dim of 766 days (between 2023-03-29 & 2025-05-02)
    if len(data_23_25_g.time) != len(dschg_darray_g.time):
        print(f"Error: [8.1] Both arrays are assumed to match in size. Please verify the code.")
    if len(dschg_darray_g.time) != time_dim_len:
        dschg_darray_g = dschg_darray_g.reindex(time=time_dim) # asia doesn't -> fill the time between 2024 & 2025 with NaN
        print(f"[10] DataArray of continent {i_continent} was reindexed") # missing values seem to be nan [tested]

    ## Section 11 : Append continent DataArrays to list
    # grdc
    runoff_darrays_g[i_continent] = dschg_darray_g
    geox_darrays_g[i_continent] = geox_darray_g
    geoy_darrays_g[i_continent] = geoy_darray_g
    area_darrays_g[i_continent] = area_darray_g
    river_darrays_g[i_continent] = river_darray_g
    country_darrays_g[i_continent] = country_darray_g
    # swot
    dschg_darrays_s[i_continent] = dschg_darray_s
    geox_darrays_s[i_continent] = geox_darray_s
    geoy_darrays_s[i_continent] = geoy_darray_s
    id_darrays_s[i_continent] = id_darray_s
    w_darrays_s[i_continent] = w_darray_s
    area_darrays_s[i_continent] = area_darray_s
    river_darrays_s[i_continent] = river_darray_s

# end of the (for i_continent in continent_list) loop 

## Section 12 : Concathenate continent DataArrays. These are individual DataArrays that contain values for all continents
# grdc
runoff_global_g = xr.concat(list(runoff_darrays_g.values()), dim='id') 
geox_global_g = xr.concat(list(geox_darrays_g.values()), dim='id')
geoy_global_g = xr.concat(list(geoy_darrays_g.values()), dim='id')
area_global_g = xr.concat(list(area_darrays_g.values()), dim='id')
river_global_g = xr.concat(list(river_darrays_g.values()), dim='id')
country_global_g = xr.concat(list(country_darrays_g.values()), dim='id')
# swot
dschg_global_s = xr.concat(list(dschg_darrays_s.values()), dim='id') 
geox_global_s = xr.concat(list(geox_darrays_s.values()), dim='id')
geoy_global_s = xr.concat(list(geoy_darrays_s.values()), dim='id')
id_global_s = xr.concat(list(id_darrays_s.values()), dim='id')
width_global_s = xr.concat(list(w_darrays_s.values()), dim='id')
area_global_s = xr.concat(list(area_darrays_s.values()), dim='id')
river_global_s = xr.concat(list(river_darrays_s.values()), dim='id')

## Section 13 : Create the DataSet and append all the variables (DataArrays) to it
# grdc
dset_global = runoff_global_g.to_dataset(name='dschg_global_g')
dset_global['geox_global_g'] = geox_global_g
dset_global['geoy_global_g'] = geoy_global_g
dset_global['area_global_g'] = area_global_g
dset_global['river_global_g'] = river_global_g
dset_global['country_global_g'] = country_global_g
# swot
dset_global['dschg_global_s'] = dschg_global_s
dset_global['geox_global_s'] = geox_global_s
dset_global['geoy_global_s'] = geoy_global_s
dset_global['id_global_s'] = id_global_s
dset_global['width_global_s'] = width_global_s
dset_global['area_global_s'] = area_global_s
dset_global['river_global_s'] = river_global_s

# Change the data type of the reaches id which is (int)
# dset_global["id_global_s"] = dset_global["id_global_s"].astype(int) # *** actually, this can't be done since a numpy array of type int cannot contain nans (they would be transformed in -9223372036854775808)

# Print
print(dset_global)

# Save to netcdf
dset_global.to_netcdf("/obs/ecastonguay/scripts/global_dset_" + version + ".nc")

Number of reaches that weren't matched due to corresp. algo: 1453 for the na continent 
128 for: no match on name 
979 for: no match on area/distance 
346 for: empty discharge
Number of reaches that weren't matched due to watershed data: 5 for the na continent.
Number of reaches that weren't matched due to corresp. algo: 347 for the af continent 
204 for: no match on name 
110 for: no match on area/distance 
33 for: empty discharge
Number of reaches that weren't matched due to watershed data: 5 for the af continent.
Number of reaches that weren't matched due to corresp. algo: 116 for the as continent 
96 for: no match on name 
13 for: no match on area/distance 
7 for: empty discharge
Number of reaches that weren't matched due to watershed data: 0 for the as continent.
Number of reaches that weren't matched due to corresp. algo: 1910 for the eu continent 
1010 for: no match on name 
609 for: no match on area/distance 
291 for: empty discharge
Number of reaches that weren't matched due t

### **V5** Correspondence based on 1) name, 2) distance & area

##### This version reformats the swot discharge data into a darray.
##### v5 : thr = [0.5, 20, 75, 50] with KDTree as distance (error)
##### v5_1 : thr = [0.5, 20, 75, 100] with geopy

In [ ]:
version = 'v5_1'
## Section 1 : Setting some variables
# 1.1 Continents
continent_list = ['na', 'af', 'as', 'eu', 'sa', 'oc']
# 1.2 Period during which SWOT has data (2023-03-29 to 2025-05-02)
swot_start = '2023-03-29'
swot_end = '2025-05-02'
# 1.3 Empty dictionnaries for the DataArrays
# grdc
runoff_darrays_g = {} 
geox_darrays_g = {}
geoy_darrays_g = {}
area_darrays_g = {}
river_darrays_g = {}
country_darrays_g = {}
# swot
dschg_darrays_s = {}
geox_darrays_s = {}
geoy_darrays_s = {}
id_darrays_s = {}
w_darrays_s = {} # from sword
area_darrays_s = {} # from sword
r_id_up_darrays_s = {} # from sword
r_id_dn_darrays_s = {} # from sword
river_darrays_s = {} # from sword

## Section 2 : Loop over continents 
for i_continent in continent_list: 
    
    ## Section 3 : Swot extraction
    # 3.1 Reading
    dir_l4 = "/obs/ecastonguay/swot_data/L4_discharge/"
    file_suffix = "_sword_v16_SOS_results_unconstrained_20230502T204408_20250502T204408_20251219T163700.nc"
    single_file_name = dir_l4 + i_continent + file_suffix 
    data_swot = nc.Dataset(single_file_name) # level 4 satellite data
    # 3.2 Extracting
    r_id_swt = data_swot.groups['reaches']['reach_id'][:]
    time_swt = data_swot.groups["consensus"]['time_int'][:]
    geox_swt = data_swot.groups["reaches"]['x'][:]
    geoy_swt = data_swot.groups["reaches"]['y'][:]
    mv_swot = data_swot.groups["consensus"]['consensus_q'].missing_value
    dschg_swt = data_swot.groups["consensus"]['consensus_q'][:] # shape (38048,). if no data at a reach, contains array([-1.e+12]). 
    assert dschg_swt.shape == r_id_swt.shape == time_swt.shape # dschhg is a list of arrays, same len as r_ids
    # 3.3 In DataArrays (for better access)
    geox_darray_full_s = xr.DataArray( 
        data=geox_swt,
        dims=["reach_id"], 
        coords=dict(reach_id=r_id_swt,),
        name="x coordinate swot")
    geoy_darray_full_s = xr.DataArray(
        data=geoy_swt,
        dims=["reach_id"], 
        coords=dict(reach_id=r_id_swt,),
        name="y coordinate swot")
    # 3.4 Reformatting the swot discharge data into a darray
    time_dim = pd.date_range(start=swot_start, end=swot_end, freq='D') # 766. ndarray-like of datetime64 data. dtype='datetime64[us].
    time_dim_len = len(time_dim)
    dschg_swt_rfm = utils.dschg_darray(dschg_swt, time_swt, r_id_swt, time_dim, mv_swot)
    dschg_darray_full_s = xr.DataArray( 
        data=dschg_swt_rfm,
        dims=["reach_id","time"], # name of the dimensions
        coords=dict(
            reach_id=r_id_swt,
            time=time_dim,
        ),
        attrs=dict(
            description="Consensus_q from SWOT",
            units="m3/s",
            missing_value=np.nan
        ),
        name="swot discharge"
    )

    ## Section 4 : Sword extraction
    # 4.1 Reading
    dir_swr = "/obs/ecastonguay/sword_data/netcdf_v16" # sword v16
    file_swr = i_continent + "_sword_v16.nc"
    path_swr = os.path.join(dir_swr,file_swr)
    data_swr = nc.Dataset(path_swr) # open the netcdf file
    # 4.2 Extracting
    r_id_swr = data_swr["reaches"]["reach_id"][:]
    facc_swr = data_swr["reaches"]["facc"][:]
    river_swr = data_swr["reaches"]["river_name"][:]
    width_swr = data_swr["reaches"]["width"][:]
    # 4.3 In DataArrays
    facc_darray_full_s = xr.DataArray(
        data=facc_swr,
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="facc sword") 
    width_darray_full_s = xr.DataArray(
        data=width_swr,
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 
    river_darray_full_s = xr.DataArray(
        data=river_swr, # contains ndarrays()
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 
    # 4.4 Pretreatment on sword names
    river_clean_full_s = []
    index_clean_full_s = [] # sword
    for idx_river, long_name in enumerate(river_darray_full_s.values):
        parts = long_name.split("; ")
        for p in parts:
            river_clean_full_s.append(re.sub(r'\s*\(.*?\)', '', p).lower().strip())
            index_clean_full_s.append(idx_river)
        
    ## Section 5 : Grdc extraction
    # 5.1 Reading
    dir_grdc_prefix = "/obs/ecastonguay/grdc_data/"
    file_nc = i_continent + ".nc"
    path_nc = os.path.join(dir_grdc_prefix,i_continent,file_nc)
    file_json = "stationbasins_" + i_continent + ".geojson"
    path_json = os.path.join(dir_grdc_prefix,i_continent,file_json)
    # 5.2 Extracting : discharge file
    data_grdc = xr.open_dataset(path_nc, engine="netcdf4") # <xarray.Dataset>
    data_23_25_g = data_grdc.sel(time=slice(swot_start,swot_end)) # DataSet. slice here includes the last day
    data_23_25_g = data_23_25_g.transpose() # swap dimensions to have (id,time) instead of (time,id)
    # 5.3 In DataArrays : discharge file
    dschg_darray_g = data_23_25_g['runoff_mean']      
    country_darray_g = data_23_25_g['country'] 
    # list of the stations_id of grdc  
    list_station_id = dschg_darray_g["id"].values 
    # 5.4 Extracting : watershed file
    data_ws = gpd.read_file(path_json) # watershed
    # pandas series
    station_id_gdf = data_ws['grdc_no'] 
    area_gdf = data_ws['area_calc']
    river_gdf = data_ws['river']
    geox_gdf = data_ws['long_pp']
    geoy_gdf = data_ws['lat_pp']
    # 5.5 In DataArrays : watershed file
    area_darray_g = xr.DataArray(
        data=area_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="Watershed areas (km2)"),
        name="watershed areas")
    river_darray_g = xr.DataArray(
        data=river_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="GRDC river names"),
        name="river names")
    geox_darray_g = xr.DataArray(
        data=geox_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="x coordinates of the GRDC station"),
        name="geo x grdc")
    geoy_darray_g = xr.DataArray(
        data=geoy_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="y coordinates of the GRDC station"),
        name="geo y grdc")  

    ## Section 6 : Empty ndarrays for swot/sword data 
    # 6.1 Swot
    # discharge
    dschg_ndarray = np.full((len(list_station_id), time_dim_len), np.nan, dtype=np.float64) # dim (id, time)
    # lon
    x_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # lat
    y_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id
    r_id_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # 6.2 Sword
    # width
    w_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # area
    facc_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id up
    r_id_up_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id down
    r_id_dn_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # river name
    river_ndarray = np.full((len(list_station_id)), '', dtype=object) # dim (id)

    ## Section 7 : Loop on stations
    station_pos = 0
    no_match_dist_area = 0 # n of stations that werent matched due to area diff. (corresp function)
    no_match_name = 0 # n of stations that werent matched due to name diff. (corresp function)
    no_match_dschg = 0 # n of stations that werent matched due to empty discharge data (corresp function)
    no_match_corresp = 0 # n of stations that werent matched due to corresp function
    no_match_ws = 0 # n of stations that werent matched due to (y,x) watershed data
    for station_id in list_station_id:
        
        # 7.1 Station id in watershed data?
        if station_id not in geox_darray_g.id.values:
            station_pos += 1
            no_match_ws += 1
            continue
        
        # 7.2 Coordinates of station
        x_found_station = geox_darray_g.sel(id=station_id).values # 4977015 -> "not all values found in index 'id'. Try setting the `method` keyword argument (example: method='nearest')."
        y_found_station = geoy_darray_g.sel(id=station_id).values # [tested]

        ## Section 8 : Corresp. based on names, then distance and area
        result, reason = utils.corresp_name_dist_area_v5(station_id, river_clean_full_s, index_clean_full_s, r_id_swr, area_darray_g, river_darray_g, facc_darray_full_s, dschg_darray_full_s, x_found_station, y_found_station, geox_darray_full_s, geoy_darray_full_s)
  
        if result is not None:
            sel_r_id, name_river_s = result 
        else: 
            if reason == 'no match on name':
                no_match_name += 1
            if reason == 'no match on distance or area':
                no_match_dist_area += 1
            if reason == 'empty dschg data':
                no_match_dschg += 1
            no_match_corresp += 1
            station_pos += 1 
            continue

        ## Section 8 : Filling empty ndarrays with sword/swot data
        # 8.1 Discharge
        dschg_ndarray[station_pos] = dschg_darray_full_s.sel(reach_id=sel_r_id).values # [tested] where there is data the space will be filled, otherwise stays NaN

        # 8.2 Coordinates
        sel_r_x = geox_darray_full_s.sel(reach_id=sel_r_id)
        sel_r_y = geoy_darray_full_s.sel(reach_id=sel_r_id)
        x_ndarray[station_pos] = sel_r_x
        y_ndarray[station_pos] = sel_r_y
        
        # 8.3 Reach id
        r_id_ndarray[station_pos] = sel_r_id

        # 8.4 Width, facc, river name (here I assume that a reach id in swot will be found in the sword database also; this assumption never caused me problems)
        w_ndarray[station_pos] = width_darray_full_s.sel(reach_id=sel_r_id)
        facc_ndarray[station_pos] = facc_darray_full_s.sel(reach_id=sel_r_id)
        river_ndarray[station_pos] = name_river_s
        
        station_pos += 1 # end of the (for station_id in list_station_id:) loop

    assert station_pos == len(list_station_id) # at the end of the station loop, these should be equal

    print(f"Number of reaches that weren't matched due to corresp. algo: {no_match_corresp} for the {i_continent} continent \n{no_match_name} for: no match on name \n{no_match_dist_area} for: no match on area/distance \n{no_match_dschg} for: empty discharge")
    print(f"Number of reaches that weren't matched due to watershed data: {no_match_ws} for the {i_continent} continent.")

    ## Section 9 : Create DataArrays for SWOT/SWORD (ndarrays ordered by station_id)
    # 9.1 Discharge
    dschg_darray_s = xr.DataArray(
        data=dschg_ndarray,
        dims=["id","time"], # name of the dimensions
        coords=dict(
            id=list_station_id,
            time=time_dim,
        ),
        attrs=dict(
            description="Consensus_q from SWOT",
            units="m3/s",
        ),
        name="swot discharge"
    )
    # 9.2 Coordinates
    if len(x_ndarray) != len(y_ndarray):
        print(f"Error: [7.2] The x and y vector of all the selected reaches' coordinates doesn't match in size")
        #break # *** activate when continent loop is added!
    # x_y_ndarray = np.hstack((x_ndarray,y_ndarray)) # doesn't work for now
    # dataarrays
    geox_darray_s = xr.DataArray(
        data=x_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="X coordinate of each selected reach in the SWOT data",
            units="degrees",
        ),
        name="swot x coordinate"
    )
    geoy_darray_s = xr.DataArray(
        data=y_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Y coordinate of each selected reach in the SWOT data",
            units="degrees",
        ),
        name="swot y coordinate"
    )
    # 9.3 Reach ids
    id_darray_s = xr.DataArray(
        data=r_id_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Id's of the selected reaches in the SWOT data"
        ),
        name="swot reach ids"
    )
    # 9.4 Width
    w_darray_s = xr.DataArray(
        data=w_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Average width for a SWOT reach (units: meters)."
        ),
        name="sword widths"
    )
    # 9.5 Area
    area_darray_s = xr.DataArray(
        data=facc_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Maximum flow accumulation value for a node or reach. Flow accumulation values are extracted from the MERIT Hydro dataset (Yamazaki et al., 2019) (units: square kilometers)."
        ),
        name="sword area" 
    )
    # 9.6 River name
    river_darray_s = xr.DataArray(
        data=river_ndarray.astype(str),
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="All river names associated with a node or reach. If there are multiple names for a node or reach they are listed in alphabetical order and separated by a semicolon."
        ),
        name="sword river name" 
    )

    ## Section 10 : Make sure all grdc DataArrays have time dim of 766 days (between 2023-03-29 & 2025-05-02)
    if len(data_23_25_g.time) != len(dschg_darray_g.time):
        print(f"Error: [8.1] Both arrays are assumed to match in size. Please verify the code.")
    if len(dschg_darray_g.time) != time_dim_len:
        dschg_darray_g = dschg_darray_g.reindex(time=time_dim) # asia doesn't -> fill the time between 2024 & 2025 with NaN
        print(f"[8.1] DataArray of continent {i_continent} was reindexed") # missing values seem to be nan [tested]

    ## Section 11 : Append continent DataArrays to list
    # grdc
    runoff_darrays_g[i_continent] = dschg_darray_g
    geox_darrays_g[i_continent] = geox_darray_g
    geoy_darrays_g[i_continent] = geoy_darray_g
    area_darrays_g[i_continent] = area_darray_g
    river_darrays_g[i_continent] = river_darray_g
    country_darrays_g[i_continent] = country_darray_g
    # swot
    dschg_darrays_s[i_continent] = dschg_darray_s
    geox_darrays_s[i_continent] = geox_darray_s
    geoy_darrays_s[i_continent] = geoy_darray_s
    id_darrays_s[i_continent] = id_darray_s
    w_darrays_s[i_continent] = w_darray_s
    area_darrays_s[i_continent] = area_darray_s
    river_darrays_s[i_continent] = river_darray_s

# end of the (for i_continent in continent_list) loop 

## Section 12 : Concathenate continent DataArrays. These are individual DataArrays that contain values for all continents
# grdc
runoff_global_g = xr.concat(list(runoff_darrays_g.values()), dim='id') 
geox_global_g = xr.concat(list(geox_darrays_g.values()), dim='id')
geoy_global_g = xr.concat(list(geoy_darrays_g.values()), dim='id')
area_global_g = xr.concat(list(area_darrays_g.values()), dim='id')
river_global_g = xr.concat(list(river_darrays_g.values()), dim='id')
country_global_g = xr.concat(list(country_darrays_g.values()), dim='id')
# swot
dschg_global_s = xr.concat(list(dschg_darrays_s.values()), dim='id') 
geox_global_s = xr.concat(list(geox_darrays_s.values()), dim='id')
geoy_global_s = xr.concat(list(geoy_darrays_s.values()), dim='id')
id_global_s = xr.concat(list(id_darrays_s.values()), dim='id')
width_global_s = xr.concat(list(w_darrays_s.values()), dim='id')
area_global_s = xr.concat(list(area_darrays_s.values()), dim='id')
river_global_s = xr.concat(list(river_darrays_s.values()), dim='id')

## Section 13 : Create the DataSet and append all the variables (DataArrays) to it
# grdc
dset_global = runoff_global_g.to_dataset(name='dschg_global_g')
dset_global['geox_global_g'] = geox_global_g
dset_global['geoy_global_g'] = geoy_global_g
dset_global['area_global_g'] = area_global_g
dset_global['river_global_g'] = river_global_g
dset_global['country_global_g'] = country_global_g
# swot
dset_global['dschg_global_s'] = dschg_global_s
dset_global['geox_global_s'] = geox_global_s
dset_global['geoy_global_s'] = geoy_global_s
dset_global['id_global_s'] = id_global_s
dset_global['width_global_s'] = width_global_s
dset_global['area_global_s'] = area_global_s
dset_global['river_global_s'] = river_global_s

# Change the data type of the reaches id which is (int)
# dset_global["id_global_s"] = dset_global["id_global_s"].astype(int) # *** actually, this can't be done since a numpy array of type int cannot contain nans (they would be transformed in -9223372036854775808)

# Print
print(dset_global)

# Save to netcdf
dset_global.to_netcdf("/obs/ecastonguay/scripts/global_dset_" + version + ".nc")

### **V7** Correspondence based on 1) name, 2) distance & area

##### Updates on geopy distances, nan mask, no limit on n of matches in fuzz, priorizing shortest distances

In [ ]:
version = 'v7_2'
## Section 1 : Setting some variables
# 1.1 Continents
continent_list = ['na', 'af', 'as', 'eu', 'sa', 'oc']
# 1.2 Period during which SWOT has data (2023-03-29 to 2025-05-02)
swot_start = '2023-03-29'
swot_end = '2025-05-02'
# 1.3 Empty dictionnaries for the DataArrays
# grdc
runoff_darrays_g = {} 
geox_darrays_g = {}
geoy_darrays_g = {}
area_darrays_g = {}
river_darrays_g = {}
country_darrays_g = {}
# swot
dschg_darrays_s = {}
geox_darrays_s = {}
geoy_darrays_s = {}
id_darrays_s = {}
w_darrays_s = {} # from sword
area_darrays_s = {} # from sword
r_id_up_darrays_s = {} # from sword
r_id_dn_darrays_s = {} # from sword
river_darrays_s = {} # from sword

## Section 2 : Loop over continents 
for i_continent in continent_list: 
    
    ## Section 3 : Swot extraction
    # 3.1 Reading
    dir_l4 = "/obs/ecastonguay/swot_data/L4_discharge/"
    file_suffix = "_sword_v16_SOS_results_unconstrained_20230502T204408_20250502T204408_20251219T163700.nc"
    single_file_name = dir_l4 + i_continent + file_suffix 
    data_swot = nc.Dataset(single_file_name) # level 4 satellite data
    # 3.2 Extracting
    r_id_swt = data_swot.groups['reaches']['reach_id'][:]
    time_swt = data_swot.groups["consensus"]['time_int'][:]
    geox_swt = data_swot.groups["reaches"]['x'][:]
    geoy_swt = data_swot.groups["reaches"]['y'][:]
    mv_swot = data_swot.groups["consensus"]['consensus_q'].missing_value
    dschg_swt = data_swot.groups["consensus"]['consensus_q'][:] # shape (38048,). if no data at a reach, contains array([-1.e+12]). 
    assert dschg_swt.shape == r_id_swt.shape == time_swt.shape # dschhg is a list of arrays, same len as r_ids
    # 3.3 In DataArrays (for better access)
    geox_darray_full_s = xr.DataArray( 
        data=geox_swt,
        dims=["reach_id"], 
        coords=dict(reach_id=r_id_swt,),
        name="x coordinate swot")
    geoy_darray_full_s = xr.DataArray(
        data=geoy_swt,
        dims=["reach_id"], 
        coords=dict(reach_id=r_id_swt,),
        name="y coordinate swot")
    # 3.4 Reformatting the swot discharge data into a darray
    time_dim = pd.date_range(start=swot_start, end=swot_end, freq='D') # 766. ndarray-like of datetime64 data. dtype='datetime64[us].
    time_dim_len = len(time_dim)
    dschg_swt_rfm = utils.dschg_darray(dschg_swt, time_swt, r_id_swt, time_dim, mv_swot)
    dschg_darray_full_s = xr.DataArray( 
        data=dschg_swt_rfm,
        dims=["reach_id","time"], # name of the dimensions
        coords=dict(
            reach_id=r_id_swt,
            time=time_dim,
        ),
        attrs=dict(
            description="Consensus_q from SWOT",
            units="m3/s",
            missing_value=np.nan
        ),
        name="swot discharge"
    )

    ## Section 4 : Sword extraction
    # 4.1 Reading
    dir_swr = "/obs/ecastonguay/sword_data/netcdf_v16" # sword v16
    file_swr = i_continent + "_sword_v16.nc"
    path_swr = os.path.join(dir_swr,file_swr)
    data_swr = nc.Dataset(path_swr) # open the netcdf file
    # 4.2 Extracting
    r_id_swr = data_swr["reaches"]["reach_id"][:]
    facc_swr = data_swr["reaches"]["facc"][:]
    river_swr = data_swr["reaches"]["river_name"][:]
    width_swr = data_swr["reaches"]["width"][:]
    # 4.3 In DataArrays
    facc_darray_full_s = xr.DataArray(
        data=facc_swr,
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="facc sword") 
    width_darray_full_s = xr.DataArray(
        data=width_swr,
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 
    river_darray_full_s = xr.DataArray(
        data=river_swr, # contains ndarrays()
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 
    # 4.4 Pretreatment on sword names
    river_clean_full_s = []
    index_clean_full_s = [] # sword
    for idx_river, long_name in enumerate(river_darray_full_s.values):
        parts = long_name.split("; ")
        for p in parts:
            river_clean_full_s.append(re.sub(r'\s*\(.*?\)', '', p).lower().strip())
            index_clean_full_s.append(idx_river)
        
    ## Section 5 : Grdc extraction
    # 5.1 Reading
    dir_grdc_prefix = "/obs/ecastonguay/grdc_data/"
    file_nc = i_continent + ".nc"
    path_nc = os.path.join(dir_grdc_prefix,i_continent,file_nc)
    file_json = "stationbasins_" + i_continent + ".geojson"
    path_json = os.path.join(dir_grdc_prefix,i_continent,file_json)
    # 5.2 Extracting : discharge file
    data_grdc = xr.open_dataset(path_nc, engine="netcdf4") # <xarray.Dataset>
    data_23_25_g = data_grdc.sel(time=slice(swot_start,swot_end)) # DataSet. slice here includes the last day
    data_23_25_g = data_23_25_g.transpose() # swap dimensions to have (id,time) instead of (time,id)
    # 5.3 In DataArrays : discharge file
    dschg_darray_g = data_23_25_g['runoff_mean']      
    country_darray_g = data_23_25_g['country'] 
    # list of the stations_id of grdc  
    list_station_id = dschg_darray_g["id"].values 
    # 5.4 Extracting : watershed file
    data_ws = gpd.read_file(path_json) # watershed
    # pandas series
    station_id_gdf = data_ws['grdc_no'] 
    area_gdf = data_ws['area_calc']
    river_gdf = data_ws['river']
    geox_gdf = data_ws['long_pp']
    geoy_gdf = data_ws['lat_pp']
    # 5.5 In DataArrays : watershed file
    area_darray_g = xr.DataArray(
        data=area_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="Watershed areas (km2)"),
        name="watershed areas")
    river_darray_g = xr.DataArray(
        data=river_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="GRDC river names"),
        name="river names")
    geox_darray_g = xr.DataArray(
        data=geox_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="x coordinates of the GRDC station"),
        name="geo x grdc")
    geoy_darray_g = xr.DataArray(
        data=geoy_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="y coordinates of the GRDC station"),
        name="geo y grdc")  

    ## Section 6 : Empty ndarrays for swot/sword data 
    # 6.1 Swot
    # discharge
    dschg_ndarray = np.full((len(list_station_id), time_dim_len), np.nan, dtype=np.float64) # dim (id, time)
    # lon
    x_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # lat
    y_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id
    r_id_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # 6.2 Sword
    # width
    w_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # area
    facc_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id up
    r_id_up_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # reach id down
    r_id_dn_ndarray = np.full((len(list_station_id)), np.nan, dtype=np.float64) # dim (id)
    # river name
    river_ndarray = np.full((len(list_station_id)), '', dtype=object) # dim (id)

    ## Section 7 : Loop on stations
    station_pos = 0
    no_match_dist_area = 0 # n of stations that werent matched due to area diff. (corresp function)
    no_match_name = 0 # n of stations that werent matched due to name diff. (corresp function)
    no_match_dschg = 0 # n of stations that werent matched due to empty discharge data (corresp function)
    no_match_corresp = 0 # n of stations that werent matched due to corresp function
    no_match_ws = 0 # n of stations that werent matched due to (y,x) watershed data
    for station_id in list_station_id:
        
        # 7.1 Station id in watershed data?
        if station_id not in geox_darray_g.id.values:
            station_pos += 1
            no_match_ws += 1
            continue
        
        # 7.2 Coordinates of station
        x_found_station = geox_darray_g.sel(id=station_id).item() # 4977015 -> "not all values found in index 'id'. Try setting the `method` keyword argument (example: method='nearest')."
        y_found_station = geoy_darray_g.sel(id=station_id).item() # [tested]

        ## Section 8 : Corresp. based on names, then distance and area
        result, reason = utils.corresp_name_dist_area_v5(station_id, river_clean_full_s, index_clean_full_s, r_id_swr, area_darray_g, river_darray_g, facc_darray_full_s, dschg_darray_full_s, x_found_station, y_found_station, geox_darray_full_s, geoy_darray_full_s)
  
        if result is not None:
            sel_r_id, name_river_s = result 
        else: 
            if reason == 'no match on name':
                no_match_name += 1
            if reason == 'no match on distance or area':
                no_match_dist_area += 1
            if reason == 'empty dschg data':
                no_match_dschg += 1
            no_match_corresp += 1
            station_pos += 1 
            continue

        ## Section 8 : Filling empty ndarrays with sword/swot data
        # 8.1 Discharge
        dschg_ndarray[station_pos] = dschg_darray_full_s.sel(reach_id=sel_r_id).values # [tested] where there is data the space will be filled, otherwise stays NaN

        # 8.2 Coordinates
        sel_r_x = geox_darray_full_s.sel(reach_id=sel_r_id)
        sel_r_y = geoy_darray_full_s.sel(reach_id=sel_r_id)
        x_ndarray[station_pos] = sel_r_x
        y_ndarray[station_pos] = sel_r_y
        
        # 8.3 Reach id
        r_id_ndarray[station_pos] = sel_r_id

        # 8.4 Width, facc, river name (here I assume that a reach id in swot will be found in the sword database also; this assumption never caused me problems)
        w_ndarray[station_pos] = width_darray_full_s.sel(reach_id=sel_r_id)
        facc_ndarray[station_pos] = facc_darray_full_s.sel(reach_id=sel_r_id)
        river_ndarray[station_pos] = name_river_s
        
        station_pos += 1 # end of the (for station_id in list_station_id:) loop

    assert station_pos == len(list_station_id) # at the end of the station loop, these should be equal

    print(f"Number of reaches that weren't matched due to corresp. algo: {no_match_corresp} for the {i_continent} continent \n{no_match_name} for: no match on name \n{no_match_dist_area} for: no match on area/distance \n{no_match_dschg} for: empty discharge")
    print(f"Number of reaches that weren't matched due to watershed data: {no_match_ws} for the {i_continent} continent.")

    ## Section 9 : Create DataArrays for SWOT/SWORD (ndarrays ordered by station_id)
    # 9.1 Discharge
    dschg_darray_s = xr.DataArray(
        data=dschg_ndarray,
        dims=["id","time"], # name of the dimensions
        coords=dict(
            id=list_station_id,
            time=time_dim,
        ),
        attrs=dict(
            description="Consensus_q from SWOT",
            units="m3/s",
        ),
        name="swot discharge"
    )
    # 9.2 Coordinates
    if len(x_ndarray) != len(y_ndarray):
        print(f"Error: [7.2] The x and y vector of all the selected reaches' coordinates doesn't match in size")
        #break # *** activate when continent loop is added!
    # x_y_ndarray = np.hstack((x_ndarray,y_ndarray)) # doesn't work for now
    # dataarrays
    geox_darray_s = xr.DataArray(
        data=x_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="X coordinate of each selected reach in the SWOT data",
            units="degrees",
        ),
        name="swot x coordinate"
    )
    geoy_darray_s = xr.DataArray(
        data=y_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Y coordinate of each selected reach in the SWOT data",
            units="degrees",
        ),
        name="swot y coordinate"
    )
    # 9.3 Reach ids
    id_darray_s = xr.DataArray(
        data=r_id_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Id's of the selected reaches in the SWOT data"
        ),
        name="swot reach ids"
    )
    # 9.4 Width
    w_darray_s = xr.DataArray(
        data=w_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Average width for a SWOT reach (units: meters)."
        ),
        name="sword widths"
    )
    # 9.5 Area
    area_darray_s = xr.DataArray(
        data=facc_ndarray,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="Maximum flow accumulation value for a node or reach. Flow accumulation values are extracted from the MERIT Hydro dataset (Yamazaki et al., 2019) (units: square kilometers)."
        ),
        name="sword area" 
    )
    # 9.6 River name
    river_darray_s = xr.DataArray(
        data=river_ndarray.astype(str),
        dims=["id"], # name of the dimensions
        coords=dict(
            id=list_station_id,
        ),
        attrs=dict(
            description="All river names associated with a node or reach. If there are multiple names for a node or reach they are listed in alphabetical order and separated by a semicolon."
        ),
        name="sword river name" 
    )

    ## Section 10 : Make sure all grdc DataArrays have time dim of 766 days (between 2023-03-29 & 2025-05-02)
    if len(data_23_25_g.time) != len(dschg_darray_g.time):
        print(f"Error: [8.1] Both arrays are assumed to match in size. Please verify the code.")
    if len(dschg_darray_g.time) != time_dim_len:
        dschg_darray_g = dschg_darray_g.reindex(time=time_dim) # asia doesn't -> fill the time between 2024 & 2025 with NaN
        print(f"[8.1] DataArray of continent {i_continent} was reindexed") # missing values seem to be nan [tested]

    ## Section 11 : Append continent DataArrays to list
    # grdc
    runoff_darrays_g[i_continent] = dschg_darray_g
    geox_darrays_g[i_continent] = geox_darray_g
    geoy_darrays_g[i_continent] = geoy_darray_g
    area_darrays_g[i_continent] = area_darray_g
    river_darrays_g[i_continent] = river_darray_g
    country_darrays_g[i_continent] = country_darray_g
    # swot
    dschg_darrays_s[i_continent] = dschg_darray_s
    geox_darrays_s[i_continent] = geox_darray_s
    geoy_darrays_s[i_continent] = geoy_darray_s
    id_darrays_s[i_continent] = id_darray_s
    w_darrays_s[i_continent] = w_darray_s
    area_darrays_s[i_continent] = area_darray_s
    river_darrays_s[i_continent] = river_darray_s

# end of the (for i_continent in continent_list) loop 

## Section 12 : Concathenate continent DataArrays. These are individual DataArrays that contain values for all continents
# grdc
runoff_global_g = xr.concat(list(runoff_darrays_g.values()), dim='id') 
geox_global_g = xr.concat(list(geox_darrays_g.values()), dim='id')
geoy_global_g = xr.concat(list(geoy_darrays_g.values()), dim='id')
area_global_g = xr.concat(list(area_darrays_g.values()), dim='id')
river_global_g = xr.concat(list(river_darrays_g.values()), dim='id')
country_global_g = xr.concat(list(country_darrays_g.values()), dim='id')
# swot
dschg_global_s = xr.concat(list(dschg_darrays_s.values()), dim='id') 
geox_global_s = xr.concat(list(geox_darrays_s.values()), dim='id')
geoy_global_s = xr.concat(list(geoy_darrays_s.values()), dim='id')
id_global_s = xr.concat(list(id_darrays_s.values()), dim='id')
width_global_s = xr.concat(list(w_darrays_s.values()), dim='id')
area_global_s = xr.concat(list(area_darrays_s.values()), dim='id')
river_global_s = xr.concat(list(river_darrays_s.values()), dim='id')

## Section 13 : Create the DataSet and append all the variables (DataArrays) to it
# grdc
dset_global = runoff_global_g.to_dataset(name='dschg_global_g')
dset_global['geox_global_g'] = geox_global_g
dset_global['geoy_global_g'] = geoy_global_g
dset_global['area_global_g'] = area_global_g
dset_global['river_global_g'] = river_global_g
dset_global['country_global_g'] = country_global_g
# swot
dset_global['dschg_global_s'] = dschg_global_s
dset_global['geox_global_s'] = geox_global_s
dset_global['geoy_global_s'] = geoy_global_s
dset_global['id_global_s'] = id_global_s
dset_global['width_global_s'] = width_global_s
dset_global['area_global_s'] = area_global_s
dset_global['river_global_s'] = river_global_s

# Change the data type of the reaches id which is (int)
# dset_global["id_global_s"] = dset_global["id_global_s"].astype(int) # *** actually, this can't be done since a numpy array of type int cannot contain nans (they would be transformed in -9223372036854775808)

# Print
print(dset_global)

# Save to netcdf
dset_global.to_netcdf("/obs/ecastonguay/scripts/global_dset_" + version + ".nc")

## **Part II. Other tests**

#### Count number of reaches in SWOT that have discharge data but no river name (for oc continent)

In [ ]:
n_no_match_sword = 0
n_dschg_data_swot = 0

# Find all reaches that have discharge data but no river name
print(f"Total number of reaches in swot: {len(dschg_swt)}")
for index in range(len(dschg_swt)):
    # look for discharge data
    get_dschg = dschg_swt[index][:] # swot
    mask_dschg = (get_dschg != dschg_swt.missing_value)
    dschg_flt = get_dschg[mask_dschg]
    if dschg_flt_s.size != 0:
        n_dschg_data_swot += 1
        # find reach id
        r_id_found = r_id_swt[index] # find reach in swot corresponding to index
        # check if there is a name
        river_name_r = river_darray_full_s.sel(reach_id=r_id_found) # sword
        if river_name_r == 'NODATA':
            n_no_match_sword += 1

print (f"Number of reaches that have discharge data in swot: {n_dschg_data_swot}")
print (f"Number of reaches that have discharge data in swot but no river name in sword: {n_no_match_sword}")

Total number of reaches in swot: 14453
Number of reaches that have discharge data in swot: 14453
Number of reaches that have discharge data in swot but no river name in sword: 9942


#### Count number of times the name of the river in the grdc station exists in the sword river name data

In [ ]:
import re
from rapidfuzz import fuzz

## Section 1 : Setting some variables
# Continents
continent_list = ['na', 'af', 'as', 'eu', 'sa', 'oc']
# Period during which SWOT has data (2023-03-29 to 2025-05-02)
swot_start = '2023-03-29'
swot_end = '2025-05-02'
# Threshold
thr = 75
# N of matches
name_match = 0
no_match_grdc_ws = 0 # n of times the station id was not found in the watershed river name data
# empty disctionnary
dict_t_f = {}

## Section 2 : Loop over continents 
for i_continent in continent_list: 

    ## Section 3 : Sword extraction
    # 4.1 Reading
    dir_swr = "/obs/ecastonguay/sword_data/netcdf_v16" # sword v16
    file_swr = i_continent + "_sword_v16.nc"
    path_swr = os.path.join(dir_swr,file_swr)
    data_swr = nc.Dataset(path_swr) # open the netcdf file
    # 4.2 Extracting
    r_id_swr = data_swr["reaches"]["reach_id"][:]
    river_swr = data_swr["reaches"]["river_name"][:]
    # 4.3 In DataArrays
    river_darray_full_s = xr.DataArray(
        data=river_swr, # contains ndarrays()
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 

    ## Section 4 : Grdc extraction
    # 5.1 Reading
    dir_grdc_prefix = "/obs/ecastonguay/grdc_data/"
    file_nc = i_continent + ".nc"
    path_nc = os.path.join(dir_grdc_prefix,i_continent,file_nc)
    file_json = "stationbasins_" + i_continent + ".geojson"
    path_json = os.path.join(dir_grdc_prefix,i_continent,file_json)
    # 5.2 Extracting : discharge file
    data_grdc = xr.open_dataset(path_nc, engine="netcdf4") # <xarray.Dataset>
    data_23_25_g = data_grdc.sel(time=slice(swot_start,swot_end)) # DataSet. slice here includes the last day
    data_23_25_g = data_23_25_g.transpose() # swap dimensions to have (id,time) instead of (time,id)
    # 5.3 In DataArrays : discharge file
    dschg_darray_g = data_23_25_g['runoff_mean']      
    country_darray_g = data_23_25_g['country'] 
    # list of the stations_id of grdc  
    list_station_id = dschg_darray_g["id"].values 
    # 5.4 Extracting : watershed file
    data_ws = gpd.read_file(path_json) # watershed
    # pandas series
    station_id_gdf = data_ws['grdc_no'] 
    area_gdf = data_ws['area_calc']
    river_gdf = data_ws['river']
    geox_gdf = data_ws['long_pp']
    geoy_gdf = data_ws['lat_pp']
    # 5.5 In DataArrays : watershed file
    river_darray_g = xr.DataArray(
        data=river_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="GRDC river names"),
        name="river names")
    
    # empty array of falses
    t_f_match = np.full((len(list_station_id)), False) # dim (id)

    station_pos = 0 
    ## Section 5 : Compare names
    for station_id in list_station_id:
        
        if station_id not in river_darray_g.id.values:
            no_match_grdc_ws += 1
            station_pos += 1
            continue
    
        # get river name of station and put in lower case
        river_check_g = river_darray_g.sel(id=station_id).item() # to scalar
        river_lc_g = river_check_g.lower()

        # compare all sword names to see if there is a match
        for river_name_s in river_darray_full_s.values:
            river_list_s = river_name_s.split("; ")
            river_list_re_s = np.array([ re.sub(r'\s*\(.*?\)', '', river_list_s[k]) for k in range(len(river_list_s)) ])
            # lower case
            river_list_lc_s = np.array([ river_list_re_s[j].lower() for j in range(len(river_list_re_s)) ])
            # compute fuzz ratio
            ratio_list_s = np.array([fuzz.ratio(river_lc_g, river_list_lc_s[i]) for i in range(len(river_list_lc_s))])
            best_river_s = river_list_lc_s[np.argmax(ratio_list_s)]
            ratio = fuzz.ratio(river_lc_g, best_river_s)

            # threshold
            if (ratio >= thr):
                t_f_match[station_pos] = True
                name_match += 1
                break
        station_pos += 1
    # t/f arrays in dictionnary
    dict_t_f[i_continent] = t_f_match
# concathenate t/f arrays
t_f_global = np.concatenate(list(dict_t_f.values())) # in the order of the stations ids

# get vector of t/f where sword river names 
t_f_sword = np.full((len(dset_global['river_global_s'].id.values)), False) # dim (id)
pos = 0
for name_s in dset_global['river_global_s'].values:
    if name_s != '':
        t_f_sword[pos] = True
    pos += 1

assert t_f_sword.shape == t_f_global.shape

print(f"There are {name_match} name matches for a total of {len(list_station_id)-no_match_grdc_ws} stations.") # import re
from rapidfuzz import fuzz

## Section 1 : Setting some variables
# Continents
continent_list = ['na', 'af', 'as', 'eu', 'sa', 'oc']
# Period during which SWOT has data (2023-03-29 to 2025-05-02)
swot_start = '2023-03-29'
swot_end = '2025-05-02'
# Threshold
thr = 75
# N of matches
name_match = 0
no_match_grdc_ws = 0 # n of times the station id was not found in the watershed river name data
# empty disctionnary
dict_t_f = {}

## Section 2 : Loop over continents 
for i_continent in continent_list: 

    ## Section 3 : Sword extraction
    # 4.1 Reading
    dir_swr = "/obs/ecastonguay/sword_data/netcdf_v16" # sword v16
    file_swr = i_continent + "_sword_v16.nc"
    path_swr = os.path.join(dir_swr,file_swr)
    data_swr = nc.Dataset(path_swr) # open the netcdf file
    # 4.2 Extracting
    r_id_swr = data_swr["reaches"]["reach_id"][:]
    river_swr = data_swr["reaches"]["river_name"][:]
    # 4.3 In DataArrays
    river_darray_full_s = xr.DataArray(
        data=river_swr, # contains ndarrays()
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 

    ## Section 4 : Grdc extraction
    # 5.1 Reading
    dir_grdc_prefix = "/obs/ecastonguay/grdc_data/"
    file_nc = i_continent + ".nc"
    path_nc = os.path.join(dir_grdc_prefix,i_continent,file_nc)
    file_json = "stationbasins_" + i_continent + ".geojson"
    path_json = os.path.join(dir_grdc_prefix,i_continent,file_json)
    # 5.2 Extracting : discharge file
    data_grdc = xr.open_dataset(path_nc, engine="netcdf4") # <xarray.Dataset>
    data_23_25_g = data_grdc.sel(time=slice(swot_start,swot_end)) # DataSet. slice here includes the last day
    data_23_25_g = data_23_25_g.transpose() # swap dimensions to have (id,time) instead of (time,id)
    # 5.3 In DataArrays : discharge file
    dschg_darray_g = data_23_25_g['runoff_mean']      
    country_darray_g = data_23_25_g['country'] 
    # list of the stations_id of grdc  
    list_station_id = dschg_darray_g["id"].values 
    # 5.4 Extracting : watershed file
    data_ws = gpd.read_file(path_json) # watershed
    # pandas series
    station_id_gdf = data_ws['grdc_no'] 
    area_gdf = data_ws['area_calc']
    river_gdf = data_ws['river']
    geox_gdf = data_ws['long_pp']
    geoy_gdf = data_ws['lat_pp']
    # 5.5 In DataArrays : watershed file
    river_darray_g = xr.DataArray(
        data=river_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="GRDC river names"),
        name="river names")
    
    # empty array of falses
    t_f_match = np.full((len(list_station_id)), False) # dim (id)

    station_pos = 0 
    ## Section 5 : Compare names
    for station_id in list_station_id:
        
        if station_id not in river_darray_g.id.values:
            no_match_grdc_ws += 1
            station_pos += 1
            continue
    
        # get river name of station and put in lower case
        river_check_g = river_darray_g.sel(id=station_id).item() # to scalar
        river_lc_g = river_check_g.lower()

        # compare all sword names to see if there is a match
        for river_name_s in river_darray_full_s.values:
            river_list_s = river_name_s.split("; ")
            river_list_re_s = np.array([ re.sub(r'\s*\(.*?\)', '', river_list_s[k]) for k in range(len(river_list_s)) ])
            # lower case
            river_list_lc_s = np.array([ river_list_re_s[j].lower() for j in range(len(river_list_re_s)) ])
            # compute fuzz ratio
            ratio_list_s = np.array([fuzz.ratio(river_lc_g, river_list_lc_s[i]) for i in range(len(river_list_lc_s))])
            best_river_s = river_list_lc_s[np.argmax(ratio_list_s)]
            ratio = fuzz.ratio(river_lc_g, best_river_s)

            # threshold
            if (ratio >= thr):
                t_f_match[station_pos] = True
                name_match += 1
                break
        station_pos += 1
    # t/f arrays in dictionnary
    dict_t_f[i_continent] = t_f_match
# concathenate t/f arrays
t_f_global = np.concatenate(list(dict_t_f.values())) # in the order of the stations ids

# get vector of t/f where sword river names 
t_f_sword = np.full((len(dset_global['river_global_s'].id.values)), False) # dim (id)
pos = 0
for name_s in dset_global['river_global_s'].values:
    if name_s != '':
        t_f_sword[pos] = True
    pos += 1

assert t_f_sword.shape == t_f_global.shape

print(f"There are {name_match} name matches for a total of {len(list_station_id)-no_match_grdc_ws} stations.") # import re
from rapidfuzz import fuzz

## Section 1 : Setting some variables
# Continents
continent_list = ['na', 'af', 'as', 'eu', 'sa', 'oc']
# Period during which SWOT has data (2023-03-29 to 2025-05-02)
swot_start = '2023-03-29'
swot_end = '2025-05-02'
# Threshold
thr = 75
# N of matches
name_match = 0
no_match_grdc_ws = 0 # n of times the station id was not found in the watershed river name data
# empty disctionnary
dict_t_f = {}

## Section 2 : Loop over continents 
for i_continent in continent_list: 

    ## Section 3 : Sword extraction
    # 4.1 Reading
    dir_swr = "/obs/ecastonguay/sword_data/netcdf_v16" # sword v16
    file_swr = i_continent + "_sword_v16.nc"
    path_swr = os.path.join(dir_swr,file_swr)
    data_swr = nc.Dataset(path_swr) # open the netcdf file
    # 4.2 Extracting
    r_id_swr = data_swr["reaches"]["reach_id"][:]
    river_swr = data_swr["reaches"]["river_name"][:]
    # 4.3 In DataArrays
    river_darray_full_s = xr.DataArray(
        data=river_swr, # contains ndarrays()
        dims=["reach_id"], # name of the dimensions
        coords=dict(reach_id=r_id_swr,),
        name="river name sword") 

    ## Section 4 : Grdc extraction
    # 5.1 Reading
    dir_grdc_prefix = "/obs/ecastonguay/grdc_data/"
    file_nc = i_continent + ".nc"
    path_nc = os.path.join(dir_grdc_prefix,i_continent,file_nc)
    file_json = "stationbasins_" + i_continent + ".geojson"
    path_json = os.path.join(dir_grdc_prefix,i_continent,file_json)
    # 5.2 Extracting : discharge file
    data_grdc = xr.open_dataset(path_nc, engine="netcdf4") # <xarray.Dataset>
    data_23_25_g = data_grdc.sel(time=slice(swot_start,swot_end)) # DataSet. slice here includes the last day
    data_23_25_g = data_23_25_g.transpose() # swap dimensions to have (id,time) instead of (time,id)
    # 5.3 In DataArrays : discharge file
    dschg_darray_g = data_23_25_g['runoff_mean']      
    country_darray_g = data_23_25_g['country'] 
    # list of the stations_id of grdc  
    list_station_id = dschg_darray_g["id"].values 
    # 5.4 Extracting : watershed file
    data_ws = gpd.read_file(path_json) # watershed
    # pandas series
    station_id_gdf = data_ws['grdc_no'] 
    area_gdf = data_ws['area_calc']
    river_gdf = data_ws['river']
    geox_gdf = data_ws['long_pp']
    geoy_gdf = data_ws['lat_pp']
    # 5.5 In DataArrays : watershed file
    river_darray_g = xr.DataArray(
        data=river_gdf,
        dims=["id"], 
        coords=dict(id=station_id_gdf,),
        attrs=dict(description="GRDC river names"),
        name="river names")
    
    # empty array of falses
    t_f_match = np.full((len(list_station_id)), False) # dim (id)

    station_pos = 0 
    ## Section 5 : Compare names
    for station_id in list_station_id:
        
        if station_id not in river_darray_g.id.values:
            no_match_grdc_ws += 1
            station_pos += 1
            continue
    
        # get river name of station and put in lower case
        river_check_g = river_darray_g.sel(id=station_id).item() # to scalar
        river_lc_g = river_check_g.lower()

        # compare all sword names to see if there is a match
        for river_name_s in river_darray_full_s.values:
            river_list_s = river_name_s.split("; ")
            river_list_re_s = np.array([ re.sub(r'\s*\(.*?\)', '', river_list_s[k]) for k in range(len(river_list_s)) ])
            # lower case
            river_list_lc_s = np.array([ river_list_re_s[j].lower() for j in range(len(river_list_re_s)) ])
            # compute fuzz ratio
            ratio_list_s = np.array([fuzz.ratio(river_lc_g, river_list_lc_s[i]) for i in range(len(river_list_lc_s))])
            best_river_s = river_list_lc_s[np.argmax(ratio_list_s)]
            ratio = fuzz.ratio(river_lc_g, best_river_s)

            # threshold
            if (ratio >= thr):
                t_f_match[station_pos] = True
                name_match += 1
                break
        station_pos += 1
    # t/f arrays in dictionnary
    dict_t_f[i_continent] = t_f_match
# concathenate t/f arrays
t_f_global = np.concatenate(list(dict_t_f.values())) # in the order of the stations ids

# get vector of t/f where sword river names 
t_f_sword = np.full((len(dset_global['river_global_s'].id.values)), False) # dim (id)
pos = 0
for name_s in dset_global['river_global_s'].values:
    if name_s != '':
        t_f_sword[pos] = True
    pos += 1

assert t_f_sword.shape == t_f_global.shape

print(f"There are {name_match} name matches for a total of {len(dset_global['river_global_s'].id.values)-no_match_grdc_ws} stations.") # There are 3546 name matches for a total of 5174 stations. 69% (12m) -> 1628 no match

AssertionError: 